# Лабораторная работа 9

**Тема:** численное решение обыкновенных дифференциальных уравнений (задача Коши).

**Выполнил:** Ипатенков Григорий Александрович, ИВТ-4, вариант 6

**Вариант 6:** $a = 0.4,\; b = 0.9,\; l = 1.7,\; n = 2$

$$y' = (-1)^n a\,y + b\,x^2 + l \;=\; 0.4\,y + 0.9\,x^2 + 1.7, \qquad y(0) = 1, \qquad x \in [0, 2], \quad h = 0.2.$$

**Задание:**
1. Решить задачу Коши аналитически, а также численно методами Эйлера и Рунге–Кутты IV порядка.
2. Получить явный вид аналитического решения для своих параметров.
3. Представить итог в виде сравнительной таблицы аналитического и численных решений.
4. Вывести графики аналитического решения и интерполяционных кривых обоих численных методов. Сделать выводы.

## Импорт и параметры

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import BarycentricInterpolator

plt.rcParams["figure.dpi"] = 100

In [ ]:
ASK_INPUT = False   # True — ввод параметров с клавиатуры при запуске


def ask(prompt, default, cast=float):
    if not ASK_INPUT:
        return default
    s = input(f"{prompt} [{default}]: ").strip()
    return cast(s) if s else default


a = ask("a", 0.4)
b = ask("b", 0.9)
l = ask("l", 1.7)  # noqa: E741 — обозначение из задания
n = ask("n", 2, int)
X0, Y0 = 0.0, 1.0
X_END = ask("Правый конец отрезка", 2.0)
H = ask("Шаг h", 0.2)

lam = (-1) ** n * a              # коэффициент при y


def f(x, y):
    return lam * y + b * x ** 2 + l


print(f"y' = {lam:+g}·y + {b}·x² + {l},  y({X0}) = {Y0},  x ∈ [{X0}, {X_END}],  h = {H}")

## 1. Аналитическое решение

Уравнение **линейное** ($y$ и $y'$ входят в первой степени), с **постоянными коэффициентами**,
а неоднородность $0.9x^2 + 1.7$ — многочлен. Поэтому оно решается в квадратурах.
Общее решение равно сумме общего решения однородного уравнения и частного решения неоднородного: $y = \bar y + y^*$.

**Шаг 1. Однородное уравнение** $y' - 0.4\,y = 0$. Подстановка Эйлера $y = Ce^{\lambda x}$ даёт
характеристическое уравнение $\lambda - 0.4 = 0$, откуда $\lambda = (-1)^2 \cdot 0.4 = 0.4$. Значит, $\bar y = Ce^{0.4x}$.

**Шаг 2. Частное решение** ищем в виде многочлена той же степени, что и неоднородность:
$y^* = Kx^2 + Lx + M$, $(y^*)' = 2Kx + L$. Подставляем в уравнение:

$$2Kx + L - 0.4\,(Kx^2 + Lx + M) = 0.9x^2 + 1.7.$$

Приравниваем коэффициенты при одинаковых степенях $x$:

| степень | уравнение | решение |
|---|---|---|
| $x^2$ | $-0.4K = 0.9$ | $K = -2.25$ |
| $x^1$ | $2K - 0.4L = 0$ | $L = 5K = -11.25$ |
| $x^0$ | $L - 0.4M = 1.7$ | $M = (L - 1.7)/0.4 = -32.375$ |

**Шаг 3. Начальное условие.** $y(0) = C + M = 1$, откуда $C = 1 - M = 33.375$.

**Точное решение задачи Коши:**

$$\boxed{\,y(x) = 33.375\,e^{0.4x} - 2.25\,x^2 - 11.25\,x - 32.375\,}$$

Ниже те же коэффициенты вычисляются программой по общим формулам
$K = -b/\lambda$, $L = 2K/\lambda$, $M = (L - l)/\lambda$, $C = 1 - M$. Затем решение проверяется подстановкой в уравнение.

In [ ]:
K = -b / lam
L = 2 * K / lam
M = (L - l) / lam
C = Y0 - M


def y_exact(x):
    return C * np.exp(lam * x) + K * x ** 2 + L * x + M


def dy_exact(x):
    return C * lam * np.exp(lam * x) + 2 * K * x + L


print(f"λ = {lam},  K = {K:.6g},  L = {L:.6g},  M = {M:.6g},  C = 1 − M = {C:.6g}")
print(f"y(x) = {C:.6g}·e^({lam:g}x) {K:+.6g}·x² {L:+.6g}·x {M:+.6g}\n")

xs_check = np.linspace(X0, X_END, 1001)
residual = np.max(np.abs(dy_exact(xs_check) - f(xs_check, y_exact(xs_check))))
print(f"Проверка начального условия: y(0) = {y_exact(0.0):.12f}")
print(f"Проверка уравнения: max |y' − f(x, y)| на [0, 2] = {residual:.2e}")

## 2. Численные методы

Узлы сетки: $x_i = x_0 + i\,h$, $i = 0, 1, \dots, N$, где $N = (x_{end} - x_0)/h = 10$.

**Метод Эйлера**, формула (5). Движение по касательной, порядок точности $O(h)$:
$$y_{i+1} = y_i + h\,f(x_i, y_i).$$

**Метод Эйлера с пересчётом (Хойна)**, формула (7). Прогноз по Эйлеру, затем коррекция средним наклоном, порядок $O(h^2)$.
В задании его нет, он добавлен для сравнения:
$$y_{i+1} = y_i + \frac{h}{2}(K_1 + K_2),\quad K_1 = f(x_i, y_i),\quad K_2 = f(x_i + h,\ y_i + hK_1).$$

**Метод Рунге–Кутты IV порядка**, формула (8). Четыре наклона: в начале шага, дважды в середине и в конце. Порядок $O(h^4)$:
$$y_{i+1} = y_i + \frac{h}{6}(K_1 + 2K_2 + 2K_3 + K_4),$$
$$K_1 = f(x_i, y_i),\; K_2 = f\!\left(x_i + \tfrac h2,\ y_i + \tfrac h2 K_1\right),\; K_3 = f\!\left(x_i + \tfrac h2,\ y_i + \tfrac h2 K_2\right),\; K_4 = f(x_i + h,\ y_i + hK_3).$$

In [ ]:
def grid(h):
    steps = round((X_END - X0) / h)
    return X0 + h * np.arange(steps + 1)


def euler(f, h):
    x = grid(h)
    y = np.empty_like(x)
    y[0] = Y0
    for i in range(len(x) - 1):
        y[i + 1] = y[i] + h * f(x[i], y[i])                  # (5)
    return x, y


def heun(f, h):
    x = grid(h)
    y = np.empty_like(x)
    y[0] = Y0
    for i in range(len(x) - 1):
        k1 = f(x[i], y[i])
        k2 = f(x[i] + h, y[i] + h * k1)
        y[i + 1] = y[i] + h / 2 * (k1 + k2)                  # (7)
    return x, y


def rk4(f, h):
    x = grid(h)
    y = np.empty_like(x)
    y[0] = Y0
    for i in range(len(x) - 1):
        k1 = f(x[i], y[i])
        k2 = f(x[i] + h / 2, y[i] + h / 2 * k1)
        k3 = f(x[i] + h / 2, y[i] + h / 2 * k2)
        k4 = f(x[i] + h, y[i] + h * k3)
        y[i + 1] = y[i] + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)  # (8)
    return x, y


x, y_eu = euler(f, H)
_, y_he = heun(f, H)
_, y_rk = rk4(f, H)
y_ex = y_exact(x)

## 3. Сравнительная таблица

In [ ]:
print(f"{'x':>5}{'Точное':>13}{'Эйлер':>13}{'|ошибка|':>11}{'Хойн':>13}{'|ошибка|':>11}{'РК-4':>13}{'|ошибка|':>11}")
print("-" * 90)
for xi, ye, e1, e2, e4 in zip(x, y_ex, y_eu, y_he, y_rk):
    print(f"{xi:>5.1f}{ye:>13.6f}{e1:>13.6f}{abs(e1 - ye):>11.2e}"
          f"{e2:>13.6f}{abs(e2 - ye):>11.2e}{e4:>13.6f}{abs(e4 - ye):>11.2e}")

print(f"\nОтносительная ошибка в конце отрезка x = {X_END}:")
for name, yy in [("Эйлер", y_eu), ("Хойн", y_he), ("Рунге–Кутта 4", y_rk)]:
    print(f"  {name:<14} {abs(yy[-1] - y_ex[-1]) / abs(y_ex[-1]):.3e}")

## 4. Графики

Численный метод даёт только **таблицу** значений в узлах. Чтобы получить непрерывную кривую, как советует теория,
таблицу интерполируем **многочленом Лагранжа** по всем 11 узлам. Для устойчивости он вычисляется в барицентрической форме.

In [ ]:
xf = np.linspace(X0, X_END, 400)
interp_eu = BarycentricInterpolator(x, y_eu)
interp_rk = BarycentricInterpolator(x, y_rk)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(xf, y_exact(xf), "k", lw=2.5, label="Аналитическое решение")
ax1.plot(xf, interp_eu(xf), "r--", lw=1.5, label="Эйлер: интерполяция Лагранжа")
ax1.plot(x, y_eu, "ro", ms=5)
ax1.plot(xf, interp_rk(xf), "b:", lw=2, label="Рунге–Кутта 4: интерполяция Лагранжа")
ax1.plot(x, y_rk, "bs", ms=5, mfc="none")
ax1.set_title("Решение задачи Коши, h = 0.2")
ax1.set_xlabel("x")
ax1.set_ylabel("y(x)")
ax1.grid(alpha=0.4)
ax1.legend()

ax2.semilogy(x[1:], np.abs(y_eu - y_ex)[1:], "ro-", label="Эйлер")
ax2.semilogy(x[1:], np.abs(y_he - y_ex)[1:], "g^-", label="Хойн")
ax2.semilogy(x[1:], np.abs(y_rk - y_ex)[1:], "bs-", label="Рунге–Кутта 4")
ax2.set_title("Абсолютная ошибка в узлах")
ax2.set_xlabel("x")
ax2.set_ylabel("|y_числ − y_точн|")
ax2.grid(alpha=0.4, which="both")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
xx = np.linspace(0, 0.6, 200)
ax.plot(xx, y_exact(xx), "k", lw=2.5, label="точное решение")
for i in range(3):
    ax.plot([x[i], x[i + 1]], [y_eu[i], y_eu[i + 1]], "r-o", ms=5, label="ломаная Эйлера" if i == 0 else None)
    t = np.linspace(x[i], x[i] + H, 10)
    ax.plot(t, y_eu[i] + f(x[i], y_eu[i]) * (t - x[i]), "r:", lw=1)
ax.plot(x[:4], y_rk[:4], "bs", ms=7, mfc="none", label="узлы РК-4")
ax.set_title("Первые три шага: Эйлер идёт по касательной и отстаёт от выпуклой кривой")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid(alpha=0.4)
ax.legend()
plt.show()

## 5. Контроль точности: двойной пересчёт и порядок метода

Для одношаговых методов на практике применяют **правило двойного пересчёта**: решение считается с шагами $h$ и $h/2$.
Для метода порядка $p$ погрешность более точного результата оценивается по Рунге:
$\;R \approx \dfrac{|y_{h/2} - y_h|}{2^p - 1}$.

Кроме того, проверим порядок: при уменьшении шага вдвое ошибка Эйлера должна уменьшаться примерно в $2$ раза,
Хойна — в $4$ раза, Рунге–Кутты — в $2^4 = 16$ раз.

In [ ]:
y_end = y_exact(X_END)
print(f"Точное значение y({X_END}) = {y_end:.10f}\n")
print(f"{'Метод':<15}{'p':>3}{'y_h':>15}{'y_h/2':>15}{'оценка Рунге':>15}{'реальная ошибка':>17}")
for name, method, p in [("Эйлер", euler, 1), ("Хойн", heun, 2), ("Рунге–Кутта 4", rk4, 4)]:
    yh = method(f, H)[1][-1]
    yh2 = method(f, H / 2)[1][-1]
    print(f"{name:<15}{p:>3}{yh:>15.8f}{yh2:>15.8f}{abs(yh2 - yh) / (2 ** p - 1):>15.2e}{abs(yh2 - y_end):>17.2e}")

print(f"\n{'h':>8}{'ошибка Эйлера':>16}{'уменьш.':>9}{'ошибка Хойна':>15}{'уменьш.':>9}{'ошибка РК-4':>14}{'уменьш.':>9}")
prev = None
for h in [0.2, 0.1, 0.05, 0.025]:
    errs = [abs(m(f, h)[1][-1] - y_end) for m in (euler, heun, rk4)]
    ratios = ["" if prev is None else f"{p_ / e_:.1f}" for p_, e_ in zip(prev or errs, errs)]
    print(f"{h:>8}{errs[0]:>16.3e}{ratios[0]:>9}{errs[1]:>15.3e}{ratios[1]:>9}{errs[2]:>14.3e}{ratios[2]:>9}")
    prev = errs

## Вывод

1. Уравнение $y' = 0.4y + 0.9x^2 + 1.7$ линейное, с постоянными коэффициентами и полиномиальной неоднородностью,
   поэтому оно решено аналитически:
   $y(x) = 33.375\,e^{0.4x} - 2.25x^2 - 11.25x - 32.375$. Подстановка в уравнение и начальное условие это подтверждает.
2. **Метод Эйлера** движется по касательной к интегральной кривой. Решение выпуклое и быстро растёт, поэтому
   касательная всё время проходит ниже кривой. Ошибка накапливается от шага к шагу и к концу отрезка при $h = 0.2$
   достигает почти 9%. Порядок точности первый: уменьшение шага вдвое уменьшает ошибку лишь вдвое.
3. **Метод Рунге–Кутты IV порядка** усредняет четыре наклона на каждом шаге. При том же шаге его ошибка на
   5–6 порядков меньше, чем у Эйлера, и таблица совпадает с точным решением до 5–6 знаков.
   Уменьшение шага вдвое уменьшает ошибку примерно в 16 раз, что соответствует четвёртому порядку.
4. **Метод Хойна** (Эйлер с пересчётом) занимает промежуточное положение: второй порядок, ошибка уменьшается в 4 раза.
5. Правило двойного пересчёта даёт оценку погрешности, близкую к реальной ошибке. Значит, им можно пользоваться,
   когда точное решение неизвестно. На графике интерполяционная кривая РК-4 сливается с аналитическим решением,
   а кривая Эйлера заметно идёт ниже.